In [9]:
#@title MyDriveのマウント

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
#@title 年と月を'選択してください { run: "auto" }
selected_year = '2026' # @param ['2026', '2027', '2028'] {type:"string"}

selected_month = "03" # @param ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12'] {type:"string"}

year_month = f"{selected_year}-{selected_month}"
print( f"選択された年と月：{year_month}")

選択された年と月：2026-03


In [13]:
#@title 'MyDrive/receipts/処理結果/年-月/csv'ディレクトリ内のcsvファイルへのパスを取得

import os

# パスの指定
path = f'/content/drive/MyDrive/receipts/処理結果/{year_month}/csv'

# ファイル名のリストを取得
file_names = os.listdir(path)

# 取得したファイル名を表示
for file_name in file_names:
    print(file_name)

print()

# フルパスに変換してリストに格納
csv_paths = [os.path.join(path, f) for f in os.listdir(path)]  # 内包表記を利用
print(csv_paths)


[]


In [16]:
#@title 試しにpandasで読み込んでみる

import pandas as pd

# 先ほど取得したリストの1番目（インデックス0）を読み込む
df = pd.read_csv(csv_paths[0])

# 内容の確認
print(f"読み込んだファイル: {csv_paths[0]}")
display(df.head())

IndexError: list index out of range

In [ ]:
#@title 複数のcsvを読み込んで結合する

# 各ファイルを順番に読み込んでリストに格納
df_list = []
for path in csv_paths:
    temp_df = pd.read_csv(path)
    df_list.append(temp_df)

# すべてのDataFrameを縦方向に結合
df_concat = pd.concat(df_list, ignore_index=True)

# 結果の確認
print(f"結合したファイル数: {len(df_list)}")
print()
print(f"結合した領収書数: {len(df_concat)}")
print(len(df_concat))
print()

print('最初の3つ')
display(df_concat.head(3))
print()
print('最後の3つ')
display(df_concat.tail(3))

In [ ]:
#@title 結合したdf_concatの要素を適切な型に変換する
# 1. 日付列を datetime型に変換
df_concat['日付'] = pd.to_datetime(df_concat['日付'])

# 2. 金額・税額列から数値以外（カンマ等）を除去して整数型(int)に変換
# ※数値として読み取れない値（欠損値など）がある場合は .fillna(0) などで補完が必要
df_concat['支出金額'] = df_concat['支出金額'].replace({',': '', '¥': ''}, regex=True).astype(int)

# 変換後の型を確認
print(df_concat[['日付', '支出金額']].dtypes)
print()

display(df_concat.head())

In [ ]:
#@title 科目とコードの関係を辞書で定義

code_book = {
    "運賃": 511,
    "支払手数料": 512,
    "諸会費": 520,
    "接待交際費": 521,
    "旅行交通費": 522,
    "通信費": 523,
    "事務消耗品費": 524,
    "消耗品費": 525,
    "租税公課": 526,
    "修繕費": 529,
    "保険料": 531,
    "燃料費": 533,
    "寄付金": 535,
    "管理諸費": 536,
    "図書研修費": 537,
    "雑費": 538,
    "医薬品費": 401,
    "衛生管理費": 444,
    "福利厚生費": 430
}

In [ ]:
#@title '科目'列のインデックスを取得して、その右隣（+1）に'コード列を挿入
col_idx = df_concat.columns.get_loc('科目') + 1
df_concat.insert(col_idx, 'コード', df_concat['科目'].map(code_book))
display(df_concat.head())

In [ ]:
#@title year_month以前の日付は、year_monthに変更する。元の日付は摘要に(m月d日分)として追加する。

# year_monthをdatetime型に変換（その月の1日として設定）
target_date = pd.to_datetime(year_month + '-01')

# 条件：日付がターゲット月より前であるか判定
mask = df_concat['日付'] < target_date

# 1. 摘要の更新（「月日」の形式で追記）
df_concat.loc[mask, '摘要'] = (
    df_concat.loc[mask, '摘要'] +
    '(' + df_concat.loc[mask, '日付'].dt.month.astype(str) + '月' +
    df_concat.loc[mask, '日付'].dt.day.astype(str) + '日分)'
)

# 2. 該当する日付をyear_monthに上書き
df_concat.loc[mask, '日付'] = target_date

display(df_concat)

In [ ]:
#@title 編集したdf_concatをcsvファイルとして保存する

# 保存先のディレクトリパス
output_dir = f'/content/drive/MyDrive/receipts/処理結果/{year_month}/結合済みcsv'
file_name = f'{year_month}.csv'

# 保存先のフルパスを作成
save_path = os.path.join(output_dir, file_name)

# CSVとして保存
# index=False にすることで、DataFrameのインデックス（左端の数字列）を除外して保存できます
df_concat.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"{file_name}の保存が完了しました: {save_path}")